# LLM Semantic Parser — test harness

Tests whether an **open-weights** model can do the job §3.1 of the paper gives to the
semantic parser: read a question plus the governing provisions and emit the atomic
verification obligations and their logical relations.

**What this replaces, and what it does not.** `mrag/vine/parser.py` already holds the
prompt, the repair loop, the verifier-assignment rule and the deterministic fallback.
It needs one thing: a function that sends a prompt to a model. `local_parser.make_ask`
supplies one backed by weights you download, at temperature 0 with a fixed seed.

Everything downstream stays deterministic. A specification the model emits still has to
pass `NetworkSpec.problems()`, `instantiate()` and `Network.validate()`, and a verifier
it proposes is discarded in favour of the rule in `parser.assign_verifier`.

**The section is not chosen by the model.** `graph_parser` picks it by text similarity,
measured at 9/10 first and 10/10 within the top five on these questions. The model is
given the question and that section's provisions, exactly as §3.1 describes.

**The number to beat.** The deterministic compiler was measured on the same ten questions
and the same gold annotations:

| metric | deterministic compiler |
|---|---|
| obligation precision | 29.1% |
| obligation recall | 71.7% |
| obligation F1 | 41.3% |
| authority accuracy | 69.8% |
| dependency edges found | 9 of 43 |

Recall was acceptable; it emitted 148 obligations to find 43, and built almost no
dependencies. Those are the two things an LLM parser should fix.

**Leakage.** This notebook holds question text only. The gold annotations are not in it
and are never loaded here — scoring happens separately, against a file this notebook
cannot see.

## 1 — Install Ollama and pull the model

`gpt-oss:120b` is 65 GB and needs an 80 GB A100. If `nvidia-smi` shows 40 GB, use `gpt-oss:20b` — same family, Apache 2.0, 16 GB.

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, os, requests
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(90):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        print('ollama is up'); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ollama did not start')

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
MODEL = 'gpt-oss:120b'      # 65 GB, needs an 80 GB card
# MODEL = 'gpt-oss:20b'     # 16 GB fallback, same family and licence

!ollama pull {MODEL}
!ollama list

## 2 — Repo and graph

Needs `graph_cache.pkl` and the three reading-data files on Drive at `Beyond_RAG/vine_data/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil
REPO = '/content/Beyond_RAG_repo'
if not os.path.exists(REPO):
    !git clone https://github.com/YOUR_USER/Beyond_RAG.git {REPO}
sys.path.insert(0, REPO)
os.chdir(REPO)

SRC = '/content/drive/MyDrive/Beyond_RAG/vine_data'
os.makedirs('data/mutcd', exist_ok=True)
for f in ['graph_cache.pkl', 'FIGURE_READING_LOG.md', 'MANUAL_READING_LOG.md',
          'figure_rules.json', 'mutcd_tables.jsonl', 'kind_classifier.pkl']:
    s = os.path.join(SRC, f)
    if os.path.exists(s):
        shutil.copy(s, f'data/mutcd/{f}')
print(sorted(os.listdir('data/mutcd')))

In [ ]:
from mrag.vine.graph_parser import GraphParser
gp = GraphParser()
print(f'{len(gp.N):,} nodes, {len(gp.E):,} edges')

# The router must have the text-similarity signal. Without it, section selection
# falls back to heading overlap, which scored 3/10 instead of 9/10 -- and a wrong
# section makes a correct parse impossible, so this is checked rather than assumed.
assert hasattr(GraphParser, 'W_TEXT'), (
    'graph_parser.py is the old version. Pull the updated file before running: '
    'section routing will be wrong and the parser will be judged on it.')
print('router: text similarity present, W_TEXT =', GraphParser.W_TEXT)

## 3 — The questions

Question text only. No sections, no answers, no obligations.

In [ ]:
QUESTIONS = {
 "SAMPLE002": "A conventional-road underpass has a measured minimum vertical clearance of 14 feet 7.4 inches. The statutory maximum vehicle height is 13 feet 9 inches. The location experiences frost action, and the agency has selected a 2-inch frost allowance. The clearance is uniform across the traveled way. Determine whether an advance low-clearance sign is required, the whole-inch clearance legend recommended after applying that allowance, and whether a panel mounted on the structure could replace the advance sign. If the structure panel is added, what legend should it show? Distinguish the Standard, Guidance, and Option provisions; no advance-placement distance is requested.",
 "SAMPLE003": "Use the lowest of the three high-visibility designs in Figure 3C-1. A legally established non-intersection crossing on a road posted at 40 mph is proposed with a crosswalk width of 7 feet, measured in the direction of approaching vehicle travel. Across the roadway, the design has four white longitudinal bars arranged as two identical pairs. Each bar is 10 inches wide, the clear gap inside each pair is 10 inches, and the clear gap between the two pairs is 70 inches. There are no transverse lines. The crossing location, warning signs, and other site treatments have been resolved separately. Identify the illustrated design and decide whether these specified marking dimensions and element count satisfy the applicable Standards. Give the minimum number of pairs and the permissible between-pair gap for the stated bar dimensions.\n\n**Required visual input:** [Figure 3C-1 \u2014 original PDF page 631](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=631). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE004": "Two entrance-ramp alternatives are being checked for signal-face count and placement only. In Alternative A, the ramp has one metered general-purpose lane and a non-metered HOV bypass lane. Two faces serve the metered lane, both on one roadside pole, with the lower face mounted 4.5 feet high. In Alternative B, both lanes are metered and their greens alternate rather than always appearing simultaneously. B provides exactly one face for each lane, each on the nearest roadside pole, with no face over either lane. Other signal design and visibility provisions have been satisfied. Can the count-and-placement arrangements be accepted in both alternatives? Explain how the controlled-lane count changes the applicable rule and identify a compliant correction to B.",
 "SAMPLE005": "A reversible-lane installation, away from any toll plaza, has minimal visual clutter and is a typical application. During some periods the lane serves left turns, and all required arrow and X indications have been provided. The nominal face dimensions are 12 by 12 inches for the downward green arrow, yellow X, and red X, and 18 by 18 inches for each applicable white left-turn arrow. Check only these face dimensions. In Operating Case A, the posted and statutory speed limits, 85th-percentile speed, and operating speed are all 35 mph. In Case B all four are 40 mph. Can the same small faces be accepted in both cases? Explain the exception boundary and the nominal minimum dimensions that apply when it is unavailable.",
 "SAMPLE006": "An ordinary signalized crosswalk has a concurrent vehicular phase, with an engineered pedestrian sequence of 7 seconds WALK, 12 seconds flashing UPRAISED HAND, and 3 seconds steady UPRAISED HAND as a buffer. No preemption occurs. A pedestrian enters the crosswalk 110 feet from the countdown display. The proposed display uses 6-inch Portland orange numbers on a black opaque background immediately adjacent to the hand indication. It stays dark during WALK, begins at 15 when the flashing hand starts, and continues counting through the 3-second steady-hand buffer before reaching zero. Are its countdown operation and numeral height acceptable under their respective authorities? State what time should be counted and whether the long viewing distance creates a mandatory or recommended larger numeral size. Do not recalculate the engineered pedestrian intervals.",
 "SAMPLE007": "A two-lane, one-way public highway crosses two railroad tracks at a passive crossing. There are no active warning devices and no authorized person routinely stopping road users when a train approaches. There is no nearby signalized highway intersection, and no engineering study has justified replacing the default YIELD control with STOP control. The proposal provides only a right-side Crossbuck with a 2 TRACKS plaque and a YIELD sign on a separate support. The YIELD sign positioning and height, Crossbuck blade retroreflection, and other placement details have already been checked. Both the front and back of the Crossbuck support are left without retroreflective strips, on the claim that one-way operation permits both omissions. Are the assembly count, control choice, and support-strip omissions acceptable? State what must change and which permission can still be used.",
 "SAMPLE008": "A fabricator is preparing a paddle for an adult school crossing guard who will use it before dawn. One face says STOP; the other copies the face shown in the middle row of the PREFERRED METHOD column in Figure 6D-1. The paddle is octagonal, 24 inches across, on a rigid handle, and fully retroreflectorized. All other size and legend details are appropriate to the face being copied. The STOP face has two flashing lights, one centered above its legend and one below: the upper light is white, the lower red, and they flash at 55 flashes per minute. Can this device be accepted for the school guard? Identify the copied face, follow the school-use cross-reference, and distinguish the permitted flashing-light feature from its mandatory restrictions.\n\n**Required visual input:** [Figure 6D-1 \u2014 original PDF page 825](https://mutcd.fhwa.dot.gov/pdfs/11th_Editionr1/mutcd11theditionr1hl.pdf#page=825). Read the specified drawing; its geometry is not to scale.",
 "SAMPLE009": "An emergency-vehicle hybrid beacon has been justified for an unsignalized fire-station access. All application criteria have been satisfied, no STOP- or YIELD-controlled side-road intersection lies within 100 feet, and no grade crossing is involved. It is actuated only by authorized personnel. On each major-street approach, all relevant speed measures are 45 mph. There are two beacon faces per approach, both roadside and none over the roadway; required signs and stop lines are provided. The normal program is dark, flashing yellow, steady yellow, alternating flashing red for sufficient vehicle egress, then dark again. The yellow timing has been properly engineered. If the conflict monitor puts the beacon into flashing mode, the proposed fault program also displays alternating flashing red. Assess face count, recommended placement, normal operation, and fault operation. Which change is mandatory and which is a Guidance matter?",
 "SAMPLE010": "A three-lane, one-way street has a marked midblock crosswalk across an uncontrolled approach. An engineering study supports warning lights embedded in the pavement, applicable warning signs are installed, and no STOP, YIELD, traffic signal, or pedestrian hybrid beacon controls the crossing. The approach-side installation extends across the full crossing and provides three unidirectional units, one per lane. Each is 9 feet from the outside edge of the nearest crosswalk line and faces away from the crosswalk toward approaching traffic. No units are proposed on the departure side. Passive pedestrian detection starts flashing and ends operation after the pedestrian clears the full crosswalk. Yellow output has 55 flash periods per minute; within those periods, its separate flicker modulation repeats at 4 pulses per second. Units project 0.5 inch above the pavement. Evaluate only side-of-crosswalk placement, minimum count, offset, facing direction, actuation, flash/pulse rates, and height. Can those specified features be accepted, and which are governed by an Option or Guidance rather than a Standard?",
 "SAMPLE011": "A continuous paved, undivided, two-way arterial has exactly two traffic lanes, a 20-foot traveled way, and an ADT of 6,500 vehicles. One segment is classified rural and the adjoining segment urban. It is neither a freeway nor an expressway. An engineering study also identifies a need for edge lines on the urban segment. There is no finding that edge lines would decrease safety, and no edge-line exclusion or omission is being used. Considering only whether center lines and edge lines are called for, identify which are required by a Standard and which are recommended by Guidance on each segment. A reviewer claims that unchanged width and traffic volume make the authority identical on both segments. Is that claim correct?"
}

print(len(QUESTIONS), 'questions')

## 4 — Smoke test on one question

Run this before the full set. It shows the section chosen, how many provisions were fed
in, and whether the reply came from the model or from the deterministic fallback.

**`source` is the field that matters.** `parser.py` falls back to deterministic
compilation when the model cannot produce a valid specification three times running. If
this says fallback, the model is not working and every number afterwards would be the
deterministic compiler's again, wearing a different name.

In [ ]:
import os, time
os.environ['VINE_PARSER_MODEL'] = MODEL
from mrag.vine.local_parser import make_ask, parse_question, section_chunks

ask = make_ask(MODEL)
sid = 'SAMPLE004'
t0 = time.time()
spec, rep, secs = parse_question(gp, QUESTIONS[sid], ask=ask, k_sections=1)

print(f'section chosen : {secs[0] if secs else "none"}')
print(f'provisions fed : {len(section_chunks(gp, secs[0])) if secs else 0}')
print(f'source         : {getattr(rep, "source", "?")}      <- must not be the fallback')
print(f'attempts       : {getattr(rep, "attempts", "?")}')
print(f'obligations    : {len(spec.obligations) if spec else 0}   (gold for this one is 6)')
print(f'time           : {time.time()-t0:.0f}s\n')
if spec:
    for o in spec.obligations:
        print(f'  {o.id:4s} [{o.authority:8s}] requires={list(getattr(o,"requires",[]) or [])} '
              f'{o.claim[:88]}')
for p in getattr(rep, 'problems', [])[:3]:
    print('  problem:', p)

## 5 — Run all ten

In [ ]:
import json, time
out, t0 = {}, time.time()
for sid, q in QUESTIONS.items():
    try:
        spec, rep, secs = parse_question(gp, q, ask=ask, k_sections=1)
    except Exception as e:
        out[sid] = {'error': repr(e)}; print(f'  {sid}  FAILED {e!r}'); continue
    obs = []
    if spec:
        for o in spec.obligations:
            obs.append({'id': o.id, 'claim': o.claim,
                        'type': str(getattr(o, 'type', '')),
                        'authority': o.authority,
                        'requires': list(getattr(o, 'requires', []) or []),
                        'guard': str(getattr(o, 'guard', '') or ''),
                        'evidence_hint': list(getattr(o, 'evidence_hint', []) or [])})
    merges = []
    if spec:
        for m in (getattr(spec, 'merges', []) or []):
            merges.append({'id': getattr(m, 'id', ''),
                           'inputs': list(getattr(m, 'inputs', []) or []),
                           'type': str(getattr(m, 'type', ''))})
    out[sid] = {'sections': secs, 'obligations': obs, 'merges': merges,
                'source': getattr(rep, 'source', ''),
                'attempts': getattr(rep, 'attempts', 0),
                'problems': [str(p)[:300] for p in getattr(rep, 'problems', [])]}
    deps = sum(len(o['requires']) for o in obs)
    print(f'  {sid}  {secs[0] if secs else "-":8s} obligations {len(obs):>3}  '
          f'dependencies {deps:>3}  via {out[sid]["source"]:<14s} attempts {out[sid]["attempts"]}')

print(f'\ntotal {time.time()-t0:.0f}s')
tot = sum(len(v.get('obligations', [])) for v in out.values())
dep = sum(len(o['requires']) for v in out.values() for o in v.get('obligations', []))
fb  = sum(1 for v in out.values() if 'fallback' in str(v.get('source', '')).lower())
print(f'obligations {tot}   (deterministic compiler produced 148 for 60 gold)')
print(f'dependencies {dep}   (deterministic compiler produced 9 for 43 gold)')
print(f'questions that fell back to deterministic: {fb}/10')

json.dump(out, open('/content/llm_parser_output.json', 'w'), indent=1)
from google.colab import files
files.download('/content/llm_parser_output.json')

## 6 — Determinism check

Temperature 0 with a fixed seed must give the same specification twice. If it does not,
something in the stack is non-deterministic and the paper cannot claim reproducibility
without saying so.

In [ ]:
same = 0
for sid in list(QUESTIONS)[:4]:
    a, _, _ = parse_question(gp, QUESTIONS[sid], ask=ask, k_sections=1)
    b, _, _ = parse_question(gp, QUESTIONS[sid], ask=ask, k_sections=1)
    ka = sorted((o.claim or '')[:80] for o in (a.obligations if a else []))
    kb = sorted((o.claim or '')[:80] for o in (b.obligations if b else []))
    ok = ka == kb
    same += ok
    print(f'  {sid}  identical on a repeat run: {ok}')
print(f'\n  {same}/4 reproducible   <- must be 4/4 at temperature 0')

## 7 — What to send back

`llm_parser_output.json`, which the download in cell 5 produced.

Scoring happens against the gold annotations in a separate file, so that the key cannot
be seen while the parser is being adjusted. The metrics are Table 1's: obligation
precision, recall and F1, dependency F1, operation-type accuracy and terminal-logic
accuracy.